# 01 — Identifikasi kebutuhan sistem

Pada tahap awal, saya memastikan tujuan sistem supaya pengolahan dataset dan eksperimen model tidak melebar ke hal yang tidak dibutuhkan. Proyek ini terdiri dari dua keluaran: pemetaan satu buku ke beberapa kompetensi dan rekomendasi buku ketika pegawai membutuhkan pengembangan kompetensi tertentu.

**Batasan:** dataset lokal, 67 kompetensi dari file referensi, tanpa scraping atau Google Books API. Label `kompetensi` lama pada CSV buku hanya informasi historis, **bukan** label benar untuk evaluasi.

## 1. Persiapan dan membaca sumber data
Kode berikut menentukan lokasi folder proyek dan membaca dua sumber utama.

In [1]:
from pathlib import Path
import re, html, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'data' / 'dataset_buku_2000.csv').exists():
    ROOT = ROOT.parent
assert (ROOT / 'data' / 'dataset_buku_2000.csv').exists(), 'Buka folder utama proyek atau notebook dari subfolder notebooks.'
DATA = ROOT / 'data'
OUT = ROOT / 'outputs'
FIG = ROOT / 'figures'
OUT.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True)
books = pd.read_csv(DATA / 'dataset_buku_2000.csv', dtype=str).fillna('')
comps = pd.read_csv(DATA / 'competencies_green.csv', dtype=str).fillna('')
print(f'Buku: {len(books):,} | Kompetensi: {len(comps)}')

Buku: 973 | Kompetensi: 67


## 2. Memeriksa struktur identitas
`book_id` dan `competency_id` adalah penghubung dari satu proses ke proses berikutnya.

In [2]:
print('Kolom buku:', books.columns.tolist())
print('Kolom kompetensi:', comps.columns.tolist())
print('Book ID duplikat:', books.book_id.duplicated().sum())
print('Competency ID duplikat:', comps.competency_id.duplicated().sum())
display(books[['book_id', 'judul', 'tahun', 'author']].head(5))
display(comps[['competency_id', 'competency_name']].head(5))
assert books.book_id.notna().all() and books.book_id.nunique() == len(books)
assert comps.competency_id.nunique() == len(comps)

Kolom buku: ['book_id', 'judul', 'tahun', 'abstrak', 'kategori_buku', 'kompetensi', 'author', 'isbn', 'bahasa', 'sumber', 'source_url', 'status_pemetaan', 'jumlah_kompetensi']
Kolom kompetensi: ['competency_id', 'excel_column', 'competency_name', 'search_query']
Book ID duplikat: 0
Competency ID duplikat: 0


,book_id,judul,tahun,author
0,BK00001,Business Growth Strategy,2026,Sara Param
1,BK00002,Mastering Business Growth Strategies,2020,Eon Ranger
2,BK00003,AI-Powered Business Growth Strategies,2026,Grant Whitmore
3,BK00004,Wise Growth Strategies in Leading Family Busin...,2005,J. Schwass
4,BK00005,Analysis of growth strategies. Organic vs. ino...,2017,Tim Meierkord


,competency_id,competency_name
0,C001,Business Development
1,C002,Marketing Management
2,C003,Service and Complaint Management
3,C004,Membership Administration Management
4,C005,Strategic Purchasing


## 3. Merinci kebutuhan fungsi dan aturan penerimaan
Tabel di bawah ini menjadi daftar pengecekan ketika fungsi rekomendasi selesai dibuat.

In [3]:
kebutuhan = pd.DataFrame([
    ['Fungsi A', 'Satu buku ke banyak kompetensi', 'Daftar competency_id dan skor setiap pasangan'],
    ['Fungsi B', 'Satu kompetensi ke maksimal 10 buku', 'Urutan, judul, tahun, abstrak, kategori, author, skor'],
    ['Pencocokan', 'Semua buku × semua kompetensi', 'Matriks per model sesuai ukuran populasi'],
    ['Perbandingan', 'Lima algoritma pada input yang sama', 'Hasil terpisah dan status eksekusi tercatat'],
    ['Validasi relevansi', 'Penilai independen dan rubrik terbuka', 'Tidak memakai kolom kompetensi lama sebagai ground truth'],
], columns=['bagian', 'kebutuhan', 'kriteria_penerimaan'])
display(kebutuhan)
kebutuhan.to_csv(OUT / 'spesifikasi_kebutuhan.csv', index=False, encoding='utf-8-sig')

,bagian,kebutuhan,kriteria_penerimaan
0,Fungsi A,Satu buku ke banyak kompetensi,Daftar competency_id dan skor setiap pasangan
1,Fungsi B,Satu kompetensi ke maksimal 10 buku,"Urutan, judul, tahun, abstrak, kategori, autho..."
2,Pencocokan,Semua buku × semua kompetensi,Matriks per model sesuai ukuran populasi
3,Perbandingan,Lima algoritma pada input yang sama,Hasil terpisah dan status eksekusi tercatat
4,Validasi relevansi,Penilai independen dan rubrik terbuka,Tidak memakai kolom kompetensi lama sebagai gr...


## 4. Menghitung ruang pengujian
Jumlah pasangan adalah titik awal eksperimen. Nilai ini bukan jumlah buku yang sudah terbukti relevan.

In [4]:
jumlah_pasangan = len(books) * len(comps)
print(f'{len(books)} buku × {len(comps)} kompetensi = {jumlah_pasangan:,} pasangan/model')
assert jumlah_pasangan > 0
print('Status: kebutuhan ditetapkan; relevansi belum diuji.')

973 buku × 67 kompetensi = 65,191 pasangan/model
Status: kebutuhan ditetapkan; relevansi belum diuji.


## Kesimpulan tahap
Dua fungsi sistem dan lima algoritma memiliki standar input-output yang jelas. File `outputs/spesifikasi_kebutuhan.csv` menjadi acuan tahap berikutnya. Berikutnya saya memeriksa kondisi dataset secara lebih rinci.